In [ ]:
import pandas as pd
import numpy as np
import datetime as dt

# 1. GENERATING REALISTIC DUMMY DATA FOR TESTING
np.random.seed(42)
n_rows = 10000

data = {
    'OrderID': np.random.randint(100000, 999999, size=n_rows),
    'CustomerID': np.random.randint(1001, 1200, size=n_rows),  # ~200 unique customers
    'OrderDate': [dt.datetime(2025, 1, 1) + dt.timedelta(days=int(np.random.randint(0, 365))) for _ in range(n_rows)],
    'TotalAmount': np.random.uniform(10, 500, size=n_rows)
}

df = pd.DataFrame(data)

# 2. DATA AUDITING & CLEANING (The Pro Touch)
df['OrderDate'] = pd.to_datetime(df['OrderDate'])
initial_count = len(df)

# Drop missing values if any
df.dropna(subset=['CustomerID'], inplace=True)

# Filter out negative amounts (Returns/Cancellations)
df = df[df['TotalAmount'] > 0]
print(f"Data Auditing: Cleaned {initial_count - len(df)} anomalous rows.")

Data Auditing: Cleaned 0 anomalous rows.


In [ ]:

snapshot_date = df['OrderDate'].max() + dt.timedelta(days=1)
rfm = df.groupby('CustomerID').agg({
    'OrderDate': lambda x: (snapshot_date - x.max()).days,  # Recency
    'OrderID': 'nunique',                                   # Frequency
    'TotalAmount': 'sum'                                    # Monetary
})

rfm.rename(columns={
    'OrderDate': 'Recency',
    'OrderID': 'Frequency',
    'TotalAmount': 'Monetary'
}, inplace=True)

print("RFM Metrics Calculated Successfully:")
print(rfm.head())

RFM Metrics Calculated Successfully:
            Recency  Frequency      Monetary
CustomerID                                  
1001              2         52  13878.767851
1002              4         46  11726.371386
1003             17         48  10578.745225
1004             15         47  13416.452965
1005             11         37  10644.907229


In [ ]:

rfm['R_Score'] = pd.qcut(rfm['Recency'], q=5, labels=[5, 4, 3, 2, 1])

rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), q=5, labels=[1, 2, 3, 4, 5])
rfm['M_Score'] = pd.qcut(rfm['Monetary'], q=5, labels=[1, 2, 3, 4, 5])

rfm['RFM_Cell'] = rfm['R_Score'].astype(str) + rfm['F_Score'].astype(str) + rfm['M_Score'].astype(str)

In [ ]:
def assign_segment(row):
    r = int(row['R_Score'])
    f = int(row['F_Score'])
    m = int(row['M_Score'])
    
    if r >= 4 and f >= 4:
        return 'VIP / Champions'
    elif r >= 3 and f >= 3:
        # High value, bought recently and frequently
        return 'Loyal High-Frequency'
    elif r <= 2 and f >= 4:
        # Bought a lot earlier, but hasn't returned recently
        return 'At-Risk (High Value Churn)'
    elif r >= 4 and f <= 2:
        # New customers or low frequency recent buyers
        return 'Recent New Onboard'
    elif m >= 4 and f <= 2:
        # Spent a lot of money but only ordered once/twice
        return 'High-Ticket Occasional'
    else:
        return 'Regular / Average'

rfm['Customer_Segment'] = rfm.apply(assign_segment, axis=1)

# Final export for Power BI
portfolio_data = rfm.reset_index()
portfolio_data.to_csv('RFM_Final_Output.csv', index=False)